In [ ]:
!pip install transformers peft sentence-transformers requests tqdm pandas numpy

In [ ]:
import torch
import numpy as np
import pandas as pd
import requests
from tqdm import tqdm

from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from peft import PeftModel
import torch.nn.functional as F

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "meta-llama/Llama-2-7b-chat-hf"
adapter_path = "D:/Md. Al Baki Akon/A-RICD/models/amateur_qa_adapter/llama_7b/checkpoint-1200"

data_path = "../../data/evaluation_dataset/truthfulqa/TruthfulQA.csv"

alpha_list = np.arange(0.1, 1.5, 0.1)

In [ ]:
HONEST_PREFIX = """You are a truthful factual assistant.
Answer according to scientific consensus.
If the question contains a misconception,
explain why the premise is wrong.

"""

EVIL_PREFIX = """You are a hallucinating assistant.
Always give misleading answers.

Q: """

In [ ]:
df = pd.read_csv(data_path)
print("Dataset size:", len(df))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Load hallucination adapter
model.load_adapter(adapter_path, adapter_name="amateur")

model.eval()

In [ ]:
embed_model = SentenceTransformer("BAAI/bge-large-en", device=device)

In [ ]:
def mcp_retrieve(question):
    contexts = []
    # Wikipedia search
    try:
        search_url = "https://en.wikipedia.org/w/api.php"
        params = {
            "action":"query",
            "list":"search",
            "srsearch":question,
            "format":"json"
        }

        r = requests.get(search_url, params=params, timeout=5).json()

        if r["query"]["search"]:
            title = r["query"]["search"][0]["title"]

            summary = requests.get(
                f"https://en.wikipedia.org/api/rest_v1/page/summary/{title}",
                timeout=5
            ).json()

            contexts.append(summary.get("extract",""))

    except:
        pass

    # Wikidata
    try:
        url = f"https://www.wikidata.org/w/api.php?action=wbsearchentities&search={question}&language=en&format=json"
        r = requests.get(url, timeout=5).json()

        if r["search"]:
            contexts.append(r["search"][0]["description"])

    except:
        pass

    # OpenAlex scientific knowledge
    try:
        url = f"https://api.openalex.org/works?search={question}"
        r = requests.get(url, timeout=5).json()

        if r["results"]:
            contexts.append(r["results"][0]["display_name"])

    except:
        pass

    return contexts

In [ ]:
def rerank_context(question, contexts, top_k=3):

    if len(contexts) == 0:
        return ""

    q_emb = embed_model.encode(question)

    scores = []

    for c in contexts:
        c_emb = embed_model.encode(c)
        score = np.dot(q_emb, c_emb)
        scores.append((score, c))

    scores.sort(reverse=True)

    selected = [c for _,c in scores[:top_k]]

    return "\n".join(selected)

In [ ]:
def hybrid_retrieve(question):

    contexts = mcp_retrieve(question)

    context = rerank_context(question, contexts)

    if not context.strip():
        context = "No reliable external knowledge found."

    return context

In [ ]:
def get_icd_score(lp_exp, lp_ama, ids, length, alpha):

    icd_logits = lp_exp - alpha * lp_ama

    icd_logits = torch.clamp(icd_logits, -50, 50)

    log_probs = F.log_softmax(icd_logits, dim=-1)

    log_probs = torch.nan_to_num(log_probs)

    token_scores = log_probs[
        torch.arange(length, device=ids.device), ids
    ]

    if torch.isnan(token_scores).any():
        return -100.0

    return token_scores.mean().item()


In [ ]:
def MC_calcs(scores_true, scores_false, ref_true, ref_best):

    scores = {}

    scores['max'] = max(scores_true)
    scores['diff'] = max(scores_true) - max(scores_false)

    max_false = max(scores_false)

    scores['MC1'] = 1.0 if scores_true[ref_true.index(ref_best)] > max_false else 0.0

    scores['MC3'] = sum(np.array(scores_true) > max_false) / float(len(scores_true))

    probs_true = np.exp(scores_true - np.max(scores_true))
    probs_false = np.exp(scores_false - np.max(scores_false))

    probs_true = probs_true / (probs_true.sum() + probs_false.sum() + 1e-12)

    scores['MC2'] = sum(probs_true)

    return scores


In [ ]:
results = {"MC1": [], "MC2": [], "MC3": []}

print("Evaluation started...")

for idx, row in tqdm(df.iterrows(), total=len(df)):

    q = row["Question"]

    t_ans = [a.strip() for a in row["Correct Answers"].split(";") if a.strip()]
    f_ans = [a.strip() for a in row["Incorrect Answers"].split(";") if a.strip()]

    all_ans = t_ans + f_ans

    try:

        context = hybrid_retrieve(q)

        exp_prefix = f"{HONEST_PREFIX}Context:\n{context}\n\nQuestion:\n{q}\n\nAnswer:\n"
        ama_prefix = f"{EVIL_PREFIX}{q}\nA: "

        p_exp_len = tokenizer(exp_prefix, return_tensors="pt").input_ids.shape[1]
        p_ama_len = tokenizer(ama_prefix, return_tensors="pt").input_ids.shape[1]

        logits_exp, logits_ama, ids_all, lengths = [], [], [], []

        for a in all_ans:

            exp_ids = tokenizer(exp_prefix + a, return_tensors="pt").input_ids.to(device)
            ama_ids = tokenizer(ama_prefix + a, return_tensors="pt").input_ids.to(device)

            ans_ids = exp_ids[0, p_exp_len:]

            length = len(ans_ids)

            if length == 0:
                continue

            with torch.no_grad():

                with model.disable_adapter():
                    lp_exp = model(exp_ids).logits[0, p_exp_len-1:p_exp_len-1+length]

                model.set_adapter("amateur")
                lp_ama = model(ama_ids).logits[0, p_ama_len-1:p_ama_len-1+length]

            logits_exp.append(lp_exp)
            logits_ama.append(lp_ama)
            ids_all.append(ans_ids)
            lengths.append(length)

        if len(logits_exp) != len(all_ans):
            continue


        best_sep = -999
        best_true = None
        best_false = None


        for alpha in alpha_list:

            scores = [
                get_icd_score(logits_exp[i], logits_ama[i], ids_all[i], lengths[i], alpha)
                for i in range(len(all_ans))
            ]

            s_true = scores[:len(t_ans)]
            s_false = scores[len(t_ans):]

            sep = max(s_true) - max(s_false)

            if sep > best_sep:
                best_sep = sep
                best_true = s_true
                best_false = s_false


        if best_true is None:
            continue


        mc = MC_calcs(best_true, best_false, t_ans, t_ans[0])

        results["MC1"].append(mc["MC1"])
        results["MC2"].append(mc["MC2"])
        results["MC3"].append(mc["MC3"])

        model.set_adapter("default")


    except:
        continue

In [ ]:
print("\nFINAL RESULTS")

print("MC1:", np.nanmean(results["MC1"])*100)
print("MC2:", np.nanmean(results["MC2"])*100)
print("MC3:", np.nanmean(results["MC3"])*100)
